# 🎓 Student Marks Prediction — Without Pipeline

**Goal:** Predict a student's `final_score` based on study habits, attendance, and background.  
**Approach:** Build the ML workflow manually, step by step.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [2]:
df = pd.read_csv("data.csv")
print("Shape:", df.shape)
df.head()

Shape: (263, 10)


,study_hours,attendance_percentage,sleep_hours,previous_exam_score,internet_access,parental_education,extracurricular_activity,part_time_job,motivation_level,final_score
0,6.5,85.0,7.0,72,Yes,Graduate,Yes,No,High,78
1,4.0,70.0,6.5,58,No,High School,No,Yes,Low,54
2,7.5,92.0,8.0,85,Yes,Postgraduate,Yes,No,High,90
3,3.0,60.0,5.5,45,No,Primary,No,Yes,Low,42
4,5.5,78.0,7.0,67,Yes,High School,Yes,No,Medium,68


## 3. Quick Exploration

In [3]:
print("Columns:", df.columns.tolist())
print("\nNull values:")
print(df.isnull().sum())

Columns: ['study_hours', 'attendance_percentage', 'sleep_hours', 'previous_exam_score', 'internet_access', 'parental_education', 'extracurricular_activity', 'part_time_job', 'motivation_level', 'final_score']

Null values:
study_hours                 11
attendance_percentage        6
sleep_hours                  3
previous_exam_score          0
internet_access              0
parental_education           0
extracurricular_activity     0
part_time_job                0
motivation_level             0
final_score                  0
dtype: int64


In [4]:
df.describe()

,study_hours,attendance_percentage,sleep_hours,previous_exam_score,final_score
count,252.000000,257.000000,260.000000,263.000000,263.000000
mean,5.353175,78.243191,6.861538,67.209125,67.269962
std,1.972992,13.615686,0.916421,17.053967,20.680580
min,2.000000,50.000000,5.500000,35.000000,32.000000
25%,3.500000,68.000000,6.000000,53.500000,49.000000
50%,5.500000,78.500000,7.000000,67.000000,68.000000
75%,7.000000,90.000000,7.500000,81.000000,86.000000
max,9.000000,100.000000,8.500000,99.000000,100.000000


## 4. Separate Features and Target

In [5]:
X = df.drop(columns=["final_score"])
y = df["final_score"]

## 5. Manual Preprocessing

We need to handle:
- Missing values
- Categorical encoding
- Feature scaling

In [6]:
# Identify column types
numerical_cols   = ["study_hours", "attendance_percentage", "sleep_hours", "previous_exam_score"]
categorical_cols = ["internet_access", "parental_education", "extracurricular_activity", "part_time_job", "motivation_level"]

In [7]:
# Fill missing numerical values with mean
for col in numerical_cols:
    X[col] = X[col].fillna(X[col].mean())

# Fill missing categorical values with mode
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print("Nulls after filling:", X.isnull().sum().sum())

Nulls after filling: 0


In [8]:
# One-hot encode categorical columns
X_encoded = pd.get_dummies(X, columns=categorical_cols)

print("Shape after encoding:", X_encoded.shape)
print("Columns:", X_encoded.columns.tolist())

Shape after encoding: (263, 20)
Columns: ['study_hours', 'attendance_percentage', 'sleep_hours', 'previous_exam_score', 'internet_access_No', 'internet_access_Yes', 'parental_education_Graduate', 'parental_education_High School', 'parental_education_HighSchool', 'parental_education_Highschool', 'parental_education_Postgraduate', 'parental_education_Primary', 'extracurricular_activity_No', 'extracurricular_activity_Yes', 'part_time_job_No', 'part_time_job_Yes', 'motivation_level_High', 'motivation_level_Low', 'motivation_level_Medim', 'motivation_level_Medium']


In [9]:
# Scale numerical features
# ⚠️  We must remember which columns we scaled and with what scaler
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])

# Save column order for inference later
training_columns = X_encoded.columns.tolist()
print("Final training columns:", training_columns)

Final training columns: ['study_hours', 'attendance_percentage', 'sleep_hours', 'previous_exam_score', 'internet_access_No', 'internet_access_Yes', 'parental_education_Graduate', 'parental_education_High School', 'parental_education_HighSchool', 'parental_education_Highschool', 'parental_education_Postgraduate', 'parental_education_Primary', 'extracurricular_activity_No', 'extracurricular_activity_Yes', 'part_time_job_No', 'part_time_job_Yes', 'motivation_level_High', 'motivation_level_Low', 'motivation_level_Medim', 'motivation_level_Medium']


## 6. Train-Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:",  X_test.shape)

Train size: (210, 20)
Test size: (53, 20)


## 7. Train Model

In [11]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 8. Evaluate Model

In [12]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 0.50
RMSE : 0.76
R²   : 0.9987


## 9. Generalization Check

Are we overfitting? Compare Train R² vs Test R².

In [13]:
train_score = model.score(X_train, y_train)
test_score  = model.score(X_test,  y_test)

print(f"Train R²: {train_score:.4f}")
print(f"Test  R²: {test_score:.4f}")

# Similar values → good generalization
# Large gap       → overfitting

Train R²: 0.9997
Test  R²: 0.9987


## 10. Save Model

> **Notice:** We need to save 3 separate files just to use this model later.

In [14]:
# Save the model
joblib.dump(model,            "without_pipeline.pkl")

# Also save the scaler and column list — needed during inference
joblib.dump(scaler,           "scaler.pkl")
joblib.dump(training_columns, "training_columns.pkl")

print("Saved: without_pipeline.pkl, scaler.pkl, training_columns.pkl")

Saved: without_pipeline.pkl, scaler.pkl, training_columns.pkl
